# Лабораторная работа №4. Случайный лес

Чибугаев Илья Алексеевич, вариант 22.

## Данные и метрики

Для классификации взят [Haberman's Survival](https://archive.ics.uci.edu/dataset/43/haberman+s+survival): 306 пациентов и 3 признака. Целевой класс `1` означает смерть в течение пяти лет после операции. По возрасту, году операции и числу пораженных лимфоузлов нужно оценить этот риск.

Для регрессии взят [QSAR aquatic toxicity](https://archive.ics.uci.edu/dataset/505/qsar+aquatic+toxicity): 546 химических соединений и 8 молекулярных дескрипторов. Целевая переменная: токсичность LC50 для Daphnia magna. Такой прогноз помогает отсеивать опасные вещества до экспериментов.

Лес усредняет деревья, обученные на разных бутстрэп-выборках. Случайный набор признаков в узлах делает деревья менее похожими друг на друга.

Train и test соотносятся как 80/20, `random_state=22`. Классы при разбиении сохраняются в одинаковой пропорции. Подбор параметров идет только на train.

В классификации основной показатель: ROC-AUC. Он не зависит от выбранного порога. F1 считается для класса `1`, accuracy показывает общую долю верных ответов. В регрессии основной показатель: RMSE в единицах LC50. MAE показывает типичную абсолютную ошибку, R² показывает долю объясненной дисперсии.


## Подготовка

Общий код вынесен в `src`, чтобы не повторять его в каждой работе.

In [1]:
# Подключаю общие функции и настраиваю вывод таблиц.
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


## Проверка данных

Смотрю размеры выборок, пропуски и целевые переменные. Эти разбиения используются во всех работах.

In [2]:
# Загружаю данные и проверяю размеры выборок.
experiments = {task: TaskExperiment(4, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля класса 1: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Haberman's Survival (UCI)
По возрасту пациента, году операции и числу пораженных лимфоузлов оценить риск смерти в течение пяти лет после операции.
train=(244, 3), test=(62, 3), пропуски=0
Доля класса 1: train=0.266, test=0.258

regression: QSAR aquatic toxicity (UCI)
По молекулярным дескрипторам прогнозировать токсичность LC50 для Daphnia magna и предварительно оценивать опасность веществ для водных организмов.
train=(436, 8), test=(110, 8), пропуски=0
count    546.0000
mean       4.6584
std        1.6652
min        0.1220
25%        3.6015
50%        4.5160
75%        5.6075
max       10.0470



## Бейзлайн sklearn

Беру модель с параметрами по умолчанию. Медианный imputer оставлен как страховка, хотя пропусков в данных нет.

In [3]:
# Обучаю базовые модели sklearn.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.7258, f1=0.3704, roc_auc=0.7554; 0.06 s
regression | sklearn | baseline: rmse=1.2079, mae=0.8282, r2=0.4859; 0.05 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,sklearn,baseline,0.7258,0.3704,0.7554,NaN,NaN,NaN,0.0563
1,4,Случайный лес,regression,sklearn,baseline,NaN,NaN,NaN,1.2079,0.8282,0.4859,0.0478


## Подбор параметров

Меняю число деревьев, глубину и размер листа. Для классификации беру `sqrt` признаков в узле, для регрессии 80%. В своей версии оставляю не больше 50 деревьев ради времени расчета.

Использую пятикратную CV. Для классификации выбираю по ROC-AUC, для регрессии по RMSE.

In [4]:
# Подбираю параметры только на обучающей выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{{experiment.task}}: {{experiment.search.best_params_}}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.8065, f1=0.4545, roc_auc=0.7690; 1.85 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__max_depth': 8, 'model__min_samples_le...",0.6708,0.0957,0.0587
1,"{'model__max_depth': None, 'model__min_samples...",0.6697,0.1043,0.1117
2,"{'model__max_depth': None, 'model__min_samples...",0.6691,0.0908,0.0858
3,"{'model__max_depth': 8, 'model__min_samples_le...",0.6683,0.1058,0.0984
4,"{'model__max_depth': None, 'model__min_samples...",0.6426,0.1012,0.0927


regression | sklearn | improved: rmse=1.2108, mae=0.8508, r2=0.4834; 0.49 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_rmse,std_rmse,mean_fit_time
0,"{'model__max_depth': 8, 'model__min_samples_le...",1.1286,0.0885,0.1209
1,"{'model__max_depth': None, 'model__min_samples...",1.1289,0.0929,0.1226
2,"{'model__max_depth': 8, 'model__min_samples_le...",1.1300,0.0814,0.0790
3,"{'model__max_depth': None, 'model__min_samples...",1.1301,0.0855,0.0671
4,"{'model__max_depth': None, 'model__min_samples...",1.1350,0.0969,0.1073


## Что дала настройка

Сравниваю метрики до и после. Для ROC-AUC, F1, accuracy и R² рост означает улучшение, для RMSE и MAE наоборот.

In [5]:
# Считаю разницу метрик между базовой и настроенной моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.7554,0.7690,0.0136
f1,0.3704,0.4545,0.0842
accuracy,0.7258,0.8065,0.0806


regression


stage,baseline,improved,change
rmse,1.2079,1.2108,0.0029
mae,0.8282,0.8508,0.0225
r2,0.4859,0.4834,-0.0025


## Своя реализация

Лес собран из собственных деревьев из третьей работы. Каждое дерево получает свою выборку и свой seed.

Код лежит в `src/models/`. Сначала обучаю базовую версию на тех же данных, что и sklearn.

In [6]:
# Обучаю базовые версии собственных моделей.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.7581, f1=0.4444, roc_auc=0.7486; 0.04 s


regression | from_scratch | baseline: rmse=1.2280, mae=0.8901, r2=0.4686; 0.86 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,from_scratch,baseline,0.7581,0.4444,0.7486,NaN,NaN,NaN,0.0402
1,4,Случайный лес,regression,from_scratch,baseline,NaN,NaN,NaN,1.228,0.8901,0.4686,0.8550


## Своя модель после настройки

Переношу препроцессинг и параметры, выбранные на CV. Ограничение по числу деревьев для своего леса указано выше.

In [7]:
# Применяю найденные параметры к своим моделям.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.7742, f1=0.3636, roc_auc=0.7826; 0.09 s


regression | from_scratch | improved: rmse=1.2398, mae=0.8884, r2=0.4583; 0.49 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,sklearn,baseline,0.7258,0.3704,0.7554,NaN,NaN,NaN,0.0563
1,4,Случайный лес,classification,sklearn,improved,0.8065,0.4545,0.7690,NaN,NaN,NaN,1.8518
2,4,Случайный лес,classification,from_scratch,baseline,0.7581,0.4444,0.7486,NaN,NaN,NaN,0.0402
3,4,Случайный лес,classification,from_scratch,improved,0.7742,0.3636,0.7826,NaN,NaN,NaN,0.0861
4,4,Случайный лес,regression,sklearn,baseline,NaN,NaN,NaN,1.2079,0.8282,0.4859,0.0478
5,4,Случайный лес,regression,sklearn,improved,NaN,NaN,NaN,1.2108,0.8508,0.4834,0.4927
6,4,Случайный лес,regression,from_scratch,baseline,NaN,NaN,NaN,1.2280,0.8901,0.4686,0.8550
7,4,Случайный лес,regression,from_scratch,improved,NaN,NaN,NaN,1.2398,0.8884,0.4583,0.4892


## Результаты

Сохраняю все четыре варианта для каждой задачи. Вывод зависит от полученных чисел: если настройка не помогла, это будет видно.

In [8]:
# Сохраняю таблицу и вывожу изменение основной метрики.
output_path = ROOT / "results" / "lab04_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

def verdict(change, higher_is_better):
    if np.isclose(change, 0):
        return "не изменился"
    improved = change > 0 if higher_is_better else change < 0
    return "улучшился" if improved else "ухудшился"

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_base = frame.loc[("sklearn", "baseline"), primary]
    sklearn_best = frame.loc[("sklearn", "improved"), primary]
    custom_base = frame.loc[("from_scratch", "baseline"), primary]
    custom_best = frame.loc[("from_scratch", "improved"), primary]
    print(f"{task}, sklearn: {primary} {sklearn_base:.4f} → {sklearn_best:.4f}, "
          f"{verdict(sklearn_best - sklearn_base, higher_is_better)}.")
    print(f"{task}, своя версия: {primary} {custom_base:.4f} → {custom_best:.4f}, "
          f"{verdict(custom_best - custom_base, higher_is_better)}.")
    print(f"Параметры sklearn: {experiment.search.best_params_}")
print(f"Файл: {output_path.relative_to(ROOT)}")

classification, sklearn: roc_auc 0.7554 → 0.7690, улучшился.
classification, своя версия: roc_auc 0.7486 → 0.7826, улучшился.
Параметры sklearn: {'model__max_depth': 8, 'model__min_samples_leaf': 3, 'model__n_estimators': 80}
regression, sklearn: rmse 1.2079 → 1.2108, ухудшился.
regression, своя версия: rmse 1.2280 → 1.2398, ухудшился.
Параметры sklearn: {'model__max_depth': 8, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
Файл: results/lab04_results.csv
